# MultiModel Testing with AutoGluon

## 1. Setup and Dependencies

In [5]:
import os
import json
import pandas as pd
import numpy as np
import torch
import random
from tqdm.auto import tqdm
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import sys

if os.path.basename(os.getcwd()) == 'notebooks':
    project_root = os.path.abspath('..')
else:
    project_root = os.getcwd()

if project_root not in sys.path:
    sys.path.append(project_root)

from src.datamodule import masked_smoothed_smape

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"AutoGluon TimeSeriesPredictor imported successfully")

PyTorch version: 2.7.1+cu128
CUDA available: True
AutoGluon TimeSeriesPredictor imported successfully


## 2. Configuration

In [6]:
BASE_DIR = ".."
DATA_DIR = os.path.join(BASE_DIR, "data")
TRAIN_DIR_FILTERED = os.path.join(DATA_DIR, "train")
VAL_DIR_FILTERED = os.path.join(DATA_DIR, "val")
# TRAIN_DIR_FILTERED = os.path.join(DATA_DIR, "train_trading_only")
# VAL_DIR_FILTERED = os.path.join(DATA_DIR, "val_trading_only")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(BASE_DIR, "models")
AUTOGLUON_DIR = os.path.join(MODELS_DIR, "autogluon_quick_test") # Unique directory for quick test run

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(AUTOGLUON_DIR, exist_ok=True)

# Model configuration
TARGET_COLS = ["high", "low", "close", "volume"]
OUTPUT_CHUNK_LENGTH = 10  # Prediction length
SEED = 827

# Load a subset of assets for faster experimentation
MAX_ASSETS = 2 # Reduce assets for performance
# REVISION: Reduce LIMIT_ROWS_PER_ASSET for a very quick test run
LIMIT_ROWS_PER_ASSET = 5000 # Reduce rows per asset for performance

## 3. Load and Convert Data to AutoGluon Format

AutoGluon expects a DataFrame with columns: `[timestamp, item_id, target]`

In [7]:
def load_parquet_files(directory, max_assets=None):
    """Load all parquet files from directory."""
    all_files = [f for f in os.listdir(directory) if f.endswith('.parquet')]
    
    random.seed(SEED)
    random.shuffle(all_files)

    if max_assets:
        files = all_files[:max_assets]
    else:
        files = all_files
    
    data = {}
    for file in tqdm(files, desc="Loading parquet files"):
        asset_name = file.replace('.parquet', '')
        df = pd.read_parquet(os.path.join(directory, file))
        data[asset_name] = df
    
    return data


def convert_to_autogluon_format(data_dict, target_cols, limit_rows=None, dataset_name="data"):
    """
    Convert dict of asset DataFrames to AutoGluon TimeSeriesDataFrame format,
    including all other columns as covariates.
    """
    all_data = []
    
    # List of columns to exclude from covariates
    exclude_cols = target_cols + ['ExecutionTime']

    # to be removed
    printed_debug_for_asset = False

    for asset_name, df in tqdm(data_dict.items(), desc="Converting to AutoGluon format"):
        if limit_rows:
            df = df.iloc[-limit_rows:].copy()
        else:
            df = df.copy()
            
        # Clean and prepare timestamps
        df['ExecutionTime'] = pd.to_datetime(df['ExecutionTime'])
        if df['ExecutionTime'].dt.tz is not None:
            df['ExecutionTime'] = df['ExecutionTime'].dt.tz_localize(None)
        
        # Identify covariate columns, including 'is_trading'
        covariate_cols = [c for c in df.columns if c not in exclude_cols]
        
        # Ensure 'is_trading' is treated as a float/int if present for easier handling as a mask
        if 'is_trading' in df.columns:
            df['is_trading'] = df['is_trading'].astype(np.float32)

        for col in target_cols:
            if col in df.columns:
                # Create the base DataFrame for this target
                item_df = pd.DataFrame({
                    'timestamp': df['ExecutionTime'].values, 
                    'item_id': f"{asset_name}_{col}",  # Unique ID per asset-feature
                    'target': df[col].astype(np.float32).values
                })
                
                # Add all identified covariate columns
                for cov_col in covariate_cols:
                    if cov_col in df.columns:
                        item_df[cov_col] = df[cov_col].values
                
                # Drop rows where target or any covariate is NaN
                item_df = item_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['target'] + covariate_cols)
                all_data.append(item_df)
    
    combined_df = pd.concat(all_data, ignore_index=True)
    
    print(f"\nCombined DataFrame shape: {combined_df.shape}")
    print(f"Columns: {combined_df.columns.tolist()}")
    
    # Convert to TimeSeriesDataFrame
    ts_df = TimeSeriesDataFrame.from_data_frame(
        combined_df,
        id_column='item_id',
        timestamp_column='timestamp'
    )
    
    # Register the covariate columns with the TimeSeriesDataFrame
    if covariate_cols:
        ts_df.known_covariate_names = covariate_cols
        print(f"\nRegistered Covariates: {covariate_cols}")

    return ts_df


print("Loading data...")
train_data = load_parquet_files(TRAIN_DIR_FILTERED, max_assets=MAX_ASSETS)
val_data = load_parquet_files(VAL_DIR_FILTERED, max_assets=MAX_ASSETS)

print(f"\nLoaded {len(train_data)} training assets")
print(f"Loaded {len(val_data)} validation assets")

# Convert to AutoGluon format
train_ts = convert_to_autogluon_format(train_data, TARGET_COLS, limit_rows=LIMIT_ROWS_PER_ASSET)
val_ts = convert_to_autogluon_format(val_data, TARGET_COLS, limit_rows=LIMIT_ROWS_PER_ASSET)

print(f"\n{'='*80}")
print("AutoGluon TimeSeriesDataFrame Created (with Covariates)")
print(f"{'='*80}")
print(f"Train shape: {train_ts.shape}")
print(f"Val shape: {val_ts.shape}")

Loading data...


Loading parquet files: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 45.78it/s]



Loaded 2 training assets
Loaded 2 validation assets


Converting to AutoGluon format: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  4.67it/s]
C:\Users\merta\AppData\Local\Temp\ipykernel_35004\2549479126.py:85: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  ts_df.known_covariate_names = covariate_cols



Combined DataFrame shape: (40000, 30)
Columns: ['timestamp', 'item_id', 'target', 'hour_of_day', 'day_of_week', 'week_of_year', 'month', 'is_weekend', 'is_trading', 'time_to_delivery', 'close_lag_adj_1', 'close_delta_adj_1', 'volume_adj_1', 'close_lag_adj_2', 'close_delta_adj_2', 'volume_adj_2', 'close_lag_adj_3', 'close_delta_adj_3', 'volume_adj_3', 'close_lag_adj_4', 'close_delta_adj_4', 'volume_adj_4', 'close_lag_adj_5', 'close_delta_adj_5', 'volume_adj_5', 'close_lag_adj_6', 'close_delta_adj_6', 'volume_adj_6', 'nearest_liquid_contract_close', 'cross_contract_mean']

Registered Covariates: ['hour_of_day', 'day_of_week', 'week_of_year', 'month', 'is_weekend', 'is_trading', 'time_to_delivery', 'close_lag_adj_1', 'close_delta_adj_1', 'volume_adj_1', 'close_lag_adj_2', 'close_delta_adj_2', 'volume_adj_2', 'close_lag_adj_3', 'close_delta_adj_3', 'volume_adj_3', 'close_lag_adj_4', 'close_delta_adj_4', 'volume_adj_4', 'close_lag_adj_5', 'close_delta_adj_5', 'volume_adj_5', 'close_lag_adj

Converting to AutoGluon format: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  5.06it/s]


Combined DataFrame shape: (40000, 30)
Columns: ['timestamp', 'item_id', 'target', 'hour_of_day', 'day_of_week', 'week_of_year', 'month', 'is_weekend', 'is_trading', 'time_to_delivery', 'close_lag_adj_1', 'close_delta_adj_1', 'volume_adj_1', 'close_lag_adj_2', 'close_delta_adj_2', 'volume_adj_2', 'close_lag_adj_3', 'close_delta_adj_3', 'volume_adj_3', 'close_lag_adj_4', 'close_delta_adj_4', 'volume_adj_4', 'close_lag_adj_5', 'close_delta_adj_5', 'volume_adj_5', 'close_lag_adj_6', 'close_delta_adj_6', 'volume_adj_6', 'nearest_liquid_contract_close', 'cross_contract_mean']

Registered Covariates: ['hour_of_day', 'day_of_week', 'week_of_year', 'month', 'is_weekend', 'is_trading', 'time_to_delivery', 'close_lag_adj_1', 'close_delta_adj_1', 'volume_adj_1', 'close_lag_adj_2', 'close_delta_adj_2', 'volume_adj_2', 'close_lag_adj_3', 'close_delta_adj_3', 'volume_adj_3', 'close_lag_adj_4', 'close_delta_adj_4', 'volume_adj_4', 'close_lag_adj_5', 'close_delta_adj_5', 'volume_adj_5', 'close_lag_adj


C:\Users\merta\AppData\Local\Temp\ipykernel_35004\2549479126.py:85: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  ts_df.known_covariate_names = covariate_cols


## 4. Compare Global Models with AutoGluon

We'll use multiple hyperparameter configurations for Deep Learning models to simulate:
Zero-Shot (minimal train) vs Fine-Tuned.

In [8]:
TIME_LIMIT = 1200

print("="*80)
print("AUTOGLUON GLOBAL MODEL COMPARISON (QUICK TEST)")
print("="*80)
print(f"Prediction length: {OUTPUT_CHUNK_LENGTH}")
print(f"Time limit: {TIME_LIMIT} seconds ({TIME_LIMIT/60:.1f} minutes) for test run") 
print(f"GPU: {'Available' if torch.cuda.is_available() else 'Not available'}")
print("="*80)

# Define hyperparameters for a fast test run
hyperparameters = {
    # 1. DeepAR (Simulating Zero-Shot/Fine-Tuning)
    "DeepAR": [
        {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
        {"max_epochs": 10, "ag_args": {"name_suffix": "_FineTuned"}},
    ],
    # 2. Temporal Fusion Transformer (TFT) (Simulating Zero-Shot/Fine-Tuning)
    "TemporalFusionTransformer": [
        {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
        {"max_epochs": 10, "ag_args": {"name_suffix": "_FineTuned"}},
    ],
    # 3. PatchTST (Modern Transformer Model)
    "PatchTST": [
        {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
        {"max_epochs": 10, "ag_args": {"name_suffix": "_FineTuned"}},
    ],
    # 4. DirectTabular (PerStepTabularModel - Non-Deep Learning Baseline)
    "DirectTabular": {
        "ag_args": {"name_suffix": "TabularBaseline"}
    }
}

print("\nStarting training...")

predictor = TimeSeriesPredictor(
    prediction_length=OUTPUT_CHUNK_LENGTH,
    path=AUTOGLUON_DIR,
    target="target",
    eval_metric="MASE",
    freq="15min",  # REMOVED - this causes massive NaN padding
    verbosity=2
)

predictor.fit(
    train_data=train_ts,
    tuning_data=val_ts,
    hyperparameters=hyperparameters,
    time_limit=TIME_LIMIT,
    random_seed=SEED
)

print("\n✓ Training complete!")

Beginning AutoGluon training... Time limit = 1200s
AutoGluon will save models to 'C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\autogluon_quick_test'
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.11
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          32
GPU Count:          1
Memory Avail:       13.34 GB / 31.63 GB (42.2%)
Disk Space Avail:   299.20 GB / 928.35 GB (32.2%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'freq': '15min',
 'hyperparameters': {'DeepAR': [{'ag_args': {'name_suffix': '_ZeroShotBase'},
                                 'max_epochs': 1},
                                {'ag_args': {'name_suffix': '_FineTuned'},
                                 'max_epochs': 10}],
                     'DirectTabular': {'ag_args': {'name_suffix': 'TabularBaseline'}},
                     'PatchTST': [{'ag_args': {'n

AUTOGLUON GLOBAL MODEL COMPARISON (QUICK TEST)
Prediction length: 10
Time limit: 1200 seconds (20.0 minutes) for test run
GPU: Available

Starting training...


Provided train_data has 40000 rows, 8 time series. Median time series length is 5000 (min=5000, max=5000). 
Provided tuning_data has 40000 rows, 8 time series. Median time series length is 5000 (min=5000, max=5000). 
	Setting num_val_windows = 0 (disabling backtesting on train_data) because tuning_data is provided.

Provided data contains following columns:
	target: 'target'
	past_covariates:
		categorical:        []
		continuous (float): ['hour_of_day', 'day_of_week', 'week_of_year', 'month', 'is_weekend', 'is_trading', ...]

To learn how to fix incorrectly inferred types, please see documentation for TimeSeriesPredictor.fit

AutoGluon will gauge predictive performance using evaluation metric: 'MASE'
	This metric's sign has been flipped to adhere to being higher_is_better. The metric score can be multiplied by -1 to get the metric value.

Starting training. Start time is 2025-11-03 08:02:58
Models that will be trained: ['DirectTabularTabularBaseline', 'TemporalFusionTransformer_ZeroSh


✓ Training complete!


## 5. Evaluate Models

In [9]:
print("="*80)
print("MODEL EVALUATION (MASE - AutoGluon Metric)")
print("="*80)

# Get leaderboard
leaderboard = predictor.leaderboard(val_ts, silent=False)
print("\nLeaderboard (Ranked by MASE):")
print(leaderboard)

# Get best model info
best_model = leaderboard.iloc[0]['model']
best_score = leaderboard.iloc[0]['score_val']

print(f"\nBest Model (by MASE): {best_model}, Score (MASE): {best_score:.4f}")

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


MODEL EVALUATION (MASE - AutoGluon Metric)
                                    model    score_test     score_val  pred_time_test  pred_time_val  fit_time_marginal  fit_order
0                        WeightedEnsemble -6.058475e-14 -6.053954e-14        0.621675       0.504462           1.286171          8
1                        DeepAR_FineTuned -6.174174e-14 -6.223002e-14        0.446064       0.405280          40.015255          5
2                     DeepAR_ZeroShotBase -2.312659e-13 -2.307463e-13        0.467536       0.414368           8.589731          4
3                      PatchTST_FineTuned -2.839582e-13 -2.839582e-13        0.175611       0.099182          43.192929          7
4                   PatchTST_ZeroShotBase -8.324625e-13 -8.324625e-13        0.159774       0.110846           5.082674          6
5     TemporalFusionTransformer_FineTuned -2.311030e-05 -2.311030e-05        0.786645       0.536429         133.821663          3
6            DirectTabularTabularBaselin

## 7. Save Final Results

In [10]:
# Compile results
results = {
    'method': 'AutoGluon TimeSeriesPredictor (Covariates Enabled)',
    'model': 'DeepAR, TFT, PatchTST, DirectTabular Comparison',
    'prediction_length': OUTPUT_CHUNK_LENGTH,
    'best_model_mase': best_model,
    'best_score_mase': float(best_score),
    'num_assets': MAX_ASSETS,
    'num_features': len(TARGET_COLS),
    'num_items': len(train_ts.item_ids.unique()),
    'leaderboard_mase': leaderboard.to_dict('records')
}

# Save results
results_file = os.path.join(RESULTS_DIR, "autogluon_results_test_covariates.json")
with open(results_file, 'w') as f:
    json.dump(results, f, indent=4)

print("="*80)
print("RESULTS SAVED")
print(f"Results: {results_file}")
print("="*80)

RESULTS SAVED
Results: ..\results\autogluon_results_test_covariates.json


In [11]:
# Load the saved predictor
loaded_predictor = TimeSeriesPredictor.load(AUTOGLUON_DIR)

print("✓ Model loaded successfully")
print(f"\nModel info:")
print(f"  Path: {loaded_predictor.path}")
print(f"  Prediction length: {loaded_predictor.prediction_length}")
print(f"  Target: {loaded_predictor.target}")

# Make predictions with loaded model
test_predictions = loaded_predictor.predict(val_ts.head(100))
print(f"\nTest predictions shape: {test_predictions.shape}")
print("✓ Loaded model works correctly")

Loading predictor from path C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\autogluon_quick_test
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


✓ Model loaded successfully

Model info:
  Path: C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\autogluon_quick_test
  Prediction length: 10
  Target: target

Test predictions shape: (10, 10)
✓ Loaded model works correctly
